In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from scipy.optimize import fsolve

In [2]:
# Input datas for IPR
Qo_IPR = 250
Pwf_IPR = 2500
Pr = 3000
Pb = 2130
delta_P_skin = 200 # from DST

# Calculating Flow Efficency
F = (Pr - Pwf_IPR - delta_P_skin)/(Pr - Pwf_IPR)

In [3]:
def Vogel_IPR(Qo, Pwf, Pr, Pb):

    #Saturated Reservoir
    if Pr<=Pb:
        Qo_max = Qo/(1 - 0.2*(Pwf/Pr) - 0.8*(Pwf/Pr)**2)
        Pwf_array = np.linspace(Pr, 0, 30)
        Qo_array = Qo_max*(1 - 0.2*(Pwf_array/Pr) - 0.8*(Pwf_array/Pr)**2)

    #Understaturated Reservoir
    else:
        if Pwf>Pb:
            J = Qo/(Pr - Pwf)
            Qob = J*(Pr - Pb)
            Qo_1 = J*(Pr - np.linspace(Pr, Pb, 10))
            Qo_2 = Qob + (J*Pb/1.8)*(1 - 0.2*(np.linspace(Pb, 0, 10)/Pb) - 0.8*(np.linspace(Pb, 0, 10)/Pb)**2)
            Pwf_array = np.concatenate((np.linspace(Pr, Pb, 10), np.linspace(Pb, 0, 10)[1:]))
            Qo_array = np.concatenate((Qo_1,Qo_2[1:]))
        else:
            J = Qo/((Pr - Pb) + (Pb/1.8)*(1 - 0.2*(Pwf/Pb) - 0.8*(Pwf/Pb)**2))
            Qob = J*(Pr - Pb)
            Qo_1 = J*(Pr - np.linspace(Pr, Pb, 10))
            Qo_2 = Qob + (J*Pb/1.8)*(1 - 0.2*(np.linspace(Pb, 0, 10)/Pb) - 0.8*(np.linspace(Pb, 0, 10)/Pb)**2)
            Pwf_array = np.concatenate((np.linspace(Pr, Pb, 10), np.linspace(Pb, 0, 10)[1:]))
            Qo_array = np.concatenate((Qo_1,Qo_2[1:]))

    return Pwf_array, Qo_array

In [4]:
Pwf_array_IPR, Qo_array_IPR = Vogel_IPR(Qo_IPR, Pwf_IPR, Pr, Pb)

In [ ]:
# Interpolation
from scipy.interpolate import splrep, splev
Pwf_predictator_IPR = splrep(Qo_array_IPR, Pwf_array_IPR, k=3, s=9)
Pwf_predicted_IPR = splev(Qo_array_IPR, Pwf_predictator_IPR)